# Notebook 1: Exploring Protein Primary Structure

## Bio 4525: Structural Bioinformatics of Proteins
**Washington University in St. Louis**

---

## What is Primary Structure?

The **primary structure** of a protein is its sequence of amino acids — the linear chain of building blocks that, when folded into a 3D shape, carries out a biological function. There are 20 different amino acids, each with a unique chemical side chain that determines how the protein folds and what it can do. The primary structure is encoded directly by the DNA sequence of a gene, making it the foundation of all other levels of protein structure. Even single amino acid changes (mutations) can drastically alter protein function — as seen in sickle cell anemia, which is caused by just one amino acid substitution in hemoglobin.

## Learning Objectives

By the end of this notebook, you will be able to:

- Retrieve a protein sequence from NCBI using BioPython
- Calculate amino acid composition and frequencies
- Identify sequence motifs using pattern searching
- Create bar charts of amino acid frequency using matplotlib
- Connect sequence composition to protein function using hemoglobin as an example

**Estimated time:** 45–60 minutes

In [ ]:
# -----------------------------------------------------------------------
# SETUP: Install and import all required libraries
# Run this cell first — it only needs to run once per Colab session
# -----------------------------------------------------------------------

# Install BioPython — not pre-installed in Colab
# The --quiet flag hides the installation log to keep output tidy
!pip install biopython --quiet

# -----------------------------------------------------------------------
# Import statements — each line loads a library into our Python environment
# -----------------------------------------------------------------------

from Bio import Entrez, SeqIO  # BioPython: retrieves and parses biological sequences
from collections import Counter  # Built-in Python: efficiently counts items in a list
import pandas as pd               # Pandas: organizes data into labeled tables (DataFrames)
import matplotlib.pyplot as plt   # Matplotlib: creates charts and graphs
import numpy as np                # NumPy: handles numeric arrays (needed for chart layout)
import re                         # Regular expressions: searches text for patterns

print('All libraries imported successfully!')

---
## Section 1: Retrieving a Protein Sequence

Before we can analyze a protein, we need its sequence. Rather than copying and pasting from a website, we will retrieve it **programmatically** — directly from a database using Python. This approach is reproducible: anyone who runs this notebook will get exactly the same sequence from the same source.

### What is NCBI?

**NCBI** (National Center for Biotechnology Information) is a U.S. government database that stores biological sequences — DNA, RNA, and protein. Think of it as a massive, publicly accessible library of molecular biology data. Every sequence in NCBI has a unique **accession number** — a catalogue ID — that lets you retrieve a specific sequence reliably.

### What is BioPython's Entrez Module?

**BioPython** is a Python library designed for bioinformatics. Its `Entrez` module lets us connect to NCBI programmatically and download sequences directly into Python. Instead of searching the NCBI website in a browser, we send a request from Python and receive the sequence as structured data we can immediately work with.

### Our Target: Hemoglobin Alpha Chain

We will work with the **human hemoglobin alpha chain** (accession number `NP_000549.1`). Adult hemoglobin is a tetramer: two alpha (α) chains and two beta (β) chains. We start with the alpha chain because it is well-studied and a manageable size (142 amino acids including the initiator methionine).

In [ ]:
# -----------------------------------------------------------------------
# IMPORTANT: Replace the email below with your own WashU email address.
# NCBI requires an email so they can contact you if your queries
# cause problems. Using a fake email may get your IP address blocked.
# -----------------------------------------------------------------------
Entrez.email = 'your.email@wustl.edu'

# The accession number uniquely identifies the hemoglobin alpha chain in NCBI
alpha_accession = 'NP_000549.1'

# We wrap the network request in a try/except block.
# If something goes wrong (no internet, NCBI is down, wrong accession),
# the except block catches the error and prints a helpful message
# instead of crashing with a confusing Python traceback.
try:
    # Entrez.efetch() connects to NCBI and requests our sequence.
    # Think of this like submitting a search on the NCBI website, but in code.
    handle = Entrez.efetch(
        db='protein',          # Search the protein database (not DNA or RNA)
        id=alpha_accession,    # The accession number of the sequence we want
        rettype='fasta',       # Return the sequence in FASTA format
        retmode='text'         # Return it as plain text
    )

    # SeqIO.read() parses the FASTA text into a structured Python object.
    # The record has useful attributes:
    #   record.id          -> the accession number
    #   record.seq         -> the amino acid sequence
    #   record.description -> the full description line from NCBI
    alpha_record = SeqIO.read(handle, 'fasta')

    # Always close the connection when done
    handle.close()

    # Print information about what we retrieved
    print(f'Successfully retrieved: {alpha_record.id}')
    print(f'Description: {alpha_record.description}')
    print(f'Sequence length: {len(alpha_record.seq)} amino acids')
    print()
    print(f'First 30 amino acids: {alpha_record.seq[:30]}...')

except Exception as error:
    print(f'Error retrieving sequence: {error}')
    print()
    print('Troubleshooting tips:')
    print('  1. Replace the email address above with your own email')
    print('  2. Check your internet connection')
    print('  3. NCBI may be temporarily unavailable — try again in a few minutes')

In [ ]:
# -----------------------------------------------------------------------
# Retrieve the hemoglobin BETA chain for comparison later
# The beta chain pairs with the alpha chain in adult hemoglobin (Hb A)
# -----------------------------------------------------------------------
beta_accession = 'NP_000509.1'

try:
    handle = Entrez.efetch(
        db='protein',
        id=beta_accession,
        rettype='fasta',
        retmode='text'
    )
    beta_record = SeqIO.read(handle, 'fasta')
    handle.close()

    print(f'Successfully retrieved: {beta_record.id}')
    print(f'Description: {beta_record.description}')
    print(f'Sequence length: {len(beta_record.seq)} amino acids')
    print()
    print(f'Alpha chain: {len(alpha_record.seq)} residues')
    print(f'Beta chain:  {len(beta_record.seq)} residues')
    print()
    print(f'First 30 amino acids of beta chain: {beta_record.seq[:30]}...')
    print()
    print('Note: The two chains are similar in length but differ in sequence.')

except Exception as error:
    print(f'Error retrieving beta chain: {error}')

---
## Section 2: Amino Acid Composition

Now that we have the sequence, let us ask: **what amino acids make up hemoglobin, and in what proportions?**

### Why Does Amino Acid Composition Matter?

The amino acid composition of a protein is a fingerprint of its chemical properties:

- **Hydrophobic amino acids** (Leu, Val, Ile, Ala, Phe) tend to be buried in the protein core, away from water — they drive folding.
- **Polar and charged amino acids** (Lys, Glu, Ser, Asn) tend to be on the protein surface, interacting with the aqueous environment.
- **Amino acids with specialized chemistry** like His and Cys deserve particular attention. Histidine's side chain can act as a proton donor or acceptor near physiological pH, making it valuable for enzyme catalysis, metal coordination, and allosteric regulation — and it appears throughout protein structures, not only at active sites. Cysteine is equally versatile: its sulfhydryl group can form structural disulfide bonds, coordinate metal ions, or participate directly in catalytic reactions depending on the protein context.

Hemoglobin has a high proportion of hydrophobic residues (consistent with its tightly packed globin fold) but also key histidines that coordinate the iron atom in the heme group.

In the code below, we count each amino acid using Python's `Counter`, then organize the results into a table with **pandas**.

In [ ]:
# Convert the BioPython sequence object to a plain Python string.
# This makes it easy to work with using standard Python tools.
alpha_sequence = str(alpha_record.seq)

# Counter() tallies how many times each character appears in the string.
# Since each character is a single-letter amino acid code, this gives
# us a count for each amino acid (e.g., {'L': 18, 'V': 13, ...})
aa_counts = Counter(alpha_sequence)

print(f'Sequence length: {len(alpha_sequence)} amino acids')
print(f'Unique amino acids found: {len(aa_counts)}')
print()

# -----------------------------------------------------------------------
# Build a pandas DataFrame for clean, organized output
# A DataFrame is like a spreadsheet with labeled rows and columns
# -----------------------------------------------------------------------

aa_data = []  # Start with an empty list; we will add one row per amino acid

for amino_acid, count in aa_counts.items():
    # Calculate what percentage of the sequence this amino acid makes up
    percentage = (count / len(alpha_sequence)) * 100
    # Append a row: [single-letter code, count, percentage rounded to 2 decimals]
    aa_data.append([amino_acid, count, round(percentage, 2)])

# Create the DataFrame with descriptive column names
aa_df = pd.DataFrame(aa_data, columns=['Amino_Acid', 'Count', 'Percentage (%)'])

# Sort by count from highest to lowest so the most common appear first
aa_df = aa_df.sort_values('Count', ascending=False).reset_index(drop=True)

print('Amino Acid Composition of Hemoglobin Alpha Chain:')
print('=' * 45)
print(aa_df.to_string(index=False))

### 💭 Think About It

Look at the amino acid composition table you just generated.

1. **Which amino acids are most abundant?** Look up their chemical properties — are they hydrophobic, polar, or charged? What does this suggest about the interior of hemoglobin?

2. **Which amino acids are least common or absent?** Is tryptophan (W) or cysteine (C) present? Why might some amino acids be rare in a particular protein?

3. **Count the histidines (H).** Histidine plays a critical role in hemoglobin by coordinating the iron atom in the heme group. How many histidines are in the alpha chain? The two most important are the proximal His at position 88 (F8 helix) and the distal His at position 59 (E7 helix).

> *Hint: You can inspect specific positions using Python indexing. Try `alpha_sequence[87]` (proximal His, position 88) and `alpha_sequence[58]` (distal His, position 59). Remember — Python counts from 0, not 1!*

---
## Section 3: Visualizing Amino Acid Frequencies

Tables are useful, but a chart lets us see patterns at a glance. We will use **matplotlib** to create bar charts.

### A Quick Intro to Matplotlib

Matplotlib is the most widely used plotting library in Python. The basic workflow is:

1. Create a **figure** and **axes** using `plt.subplots()` — the figure is the whole image; the axes is the plot area
2. Draw something on the axes (bar chart, line, scatter plot, etc.)
3. Add labels: title, x-axis label, y-axis label
4. Call `plt.tight_layout()` to prevent labels from being cut off
5. Call `plt.show()` to display the chart

We will create two charts:
- **Chart 1:** Amino acid frequency in the alpha chain alone
- **Chart 2:** Side-by-side comparison of alpha vs. beta chain composition

In [ ]:
# -----------------------------------------------------------------------
# Chart 1: Amino acid frequency in the hemoglobin alpha chain
# -----------------------------------------------------------------------

# Sort amino acids alphabetically for a consistent, easy-to-read chart
amino_acids_sorted = sorted(aa_counts.keys())
counts_sorted = [aa_counts[aa] for aa in amino_acids_sorted]

# Create the figure (entire image) and axes (the plot area)
# figsize=(12, 6) sets the width and height in inches
fig, ax = plt.subplots(figsize=(12, 6))

# Draw the bar chart
# color fills the bars; edgecolor adds a visible black outline to each bar
bars = ax.bar(amino_acids_sorted, counts_sorted,
              color='steelblue', edgecolor='black', linewidth=0.5)

# Add a title and axis labels so the chart is self-explanatory
ax.set_title('Amino Acid Frequency in Hemoglobin Alpha Chain (NP_000549.1)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Amino Acid (Single-Letter Code)', fontsize=12)
ax.set_ylabel('Number of Occurrences', fontsize=12)

# Add the exact count on top of each bar so students can read exact values
for bar, count in zip(bars, counts_sorted):
    ax.text(
        bar.get_x() + bar.get_width() / 2,  # Horizontal center of the bar
        bar.get_height() + 0.2,             # Slightly above the top of the bar
        str(count),                          # The number to display
        ha='center', va='bottom', fontsize=9
    )

# Adjust spacing so nothing is cut off at the edges
plt.tight_layout()
plt.show()

In [ ]:
# -----------------------------------------------------------------------
# Chart 2: Side-by-side comparison — alpha chain vs. beta chain
# -----------------------------------------------------------------------

# Count amino acids in the beta chain using the same method as alpha
beta_sequence = str(beta_record.seq)
beta_aa_counts = Counter(beta_sequence)

# Get a sorted list of all amino acids found in either chain
# set() removes duplicates; sorted() puts them in alphabetical order
all_amino_acids = sorted(set(list(aa_counts.keys()) + list(beta_aa_counts.keys())))

# Build count lists in the same amino acid order for both chains
# .get(aa, 0) returns 0 if an amino acid is absent from a chain
alpha_counts_list = [aa_counts.get(aa, 0) for aa in all_amino_acids]
beta_counts_list = [beta_aa_counts.get(aa, 0) for aa in all_amino_acids]

# Calculate bar positions — offset alpha and beta bars to sit side by side
x_positions = np.arange(len(all_amino_acids))  # One position per amino acid group
bar_width = 0.35                                # Width of each individual bar

# Create the figure
fig, ax = plt.subplots(figsize=(14, 7))

# Draw alpha chain bars, shifted left by half a bar width
ax.bar(x_positions - bar_width / 2, alpha_counts_list, bar_width,
       label='Alpha Chain', color='steelblue', edgecolor='black', linewidth=0.5)

# Draw beta chain bars, shifted right by half a bar width
ax.bar(x_positions + bar_width / 2, beta_counts_list, bar_width,
       label='Beta Chain', color='coral', edgecolor='black', linewidth=0.5)

# Labels, title, tick marks, and legend
ax.set_title('Amino Acid Composition: Alpha Chain vs. Beta Chain',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Amino Acid (Single-Letter Code)', fontsize=12)
ax.set_ylabel('Number of Occurrences', fontsize=12)
ax.set_xticks(x_positions)
ax.set_xticklabels(all_amino_acids)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print()
print('Observation: Both chains have broadly similar amino acid compositions.')
print('This makes sense — they share the same globin fold and perform the same')
print('basic function: binding and releasing oxygen via a heme group.')

---
## Section 4: Finding Sequence Motifs

### What is a Sequence Motif?

A **sequence motif** is a short, recurring pattern of amino acids associated with a specific biological function. When the same short pattern appears in many different proteins, it often means that pattern is doing something important — binding a metal ion, forming a structural feature, or marking an active site.

Motifs can be:
- **Exact strings** — like `FPH` (Phe-Pro-His), which appears near the heme-binding pocket of globin proteins
- **Flexible patterns** — like `H.H`, meaning a histidine, then any amino acid, then another histidine

### Pattern Searching with Regular Expressions

Python's `re` module (regular expressions) lets us search for flexible patterns. A few useful regex symbols:

| Symbol | Meaning | Example |
|--------|---------|----------|
| `.` | Any single character | `H.H` matches HAH, HVH, HPH, etc. |
| `[ABC]` | Any one of these characters | `[LVI]` matches L, V, or I |
| `.{2}` | Any 2 characters in a row | `H.{2}H` matches H + any 2 AAs + H |

We will write a reusable function to search for any pattern in any sequence.

In [ ]:
def find_motif(sequence, pattern, pattern_name='motif'):
    '''
    Search for a sequence motif (pattern) in a protein sequence.

    Uses Python's re (regular expressions) module to find all occurrences
    of a pattern and return them with surrounding sequence context.

    Parameters:
    -----------
    sequence : str
        The protein sequence to search (single-letter amino acid codes)
    pattern : str
        The pattern to search for. Can be a plain string (e.g., 'FPH')
        or a regex pattern (e.g., 'H.H' finds His-any-His)
    pattern_name : str
        A descriptive name for the motif, used in printed output

    Returns:
    --------
    list of tuples: each tuple is (position, matched_seq, context)
        position    : 1-based position (biologists count from 1, not 0)
        matched_seq : the exact amino acids that matched the pattern
        context     : 5 residues on each side for biological context
    '''
    matches = []  # Collect all matches here

    # re.finditer() scans the sequence and returns each match one at a time
    for match in re.finditer(pattern, str(sequence)):
        start_pos = match.start()  # 0-based index where the match begins
        end_pos = match.end()      # 0-based index where the match ends

        # Extract 5 residues before and after the match for context
        # max() and min() prevent going past the ends of the sequence
        context_start = max(0, start_pos - 5)
        context_end = min(len(sequence), end_pos + 5)
        context = str(sequence)[context_start:context_end]

        # Store with 1-based position (add 1 to convert from Python's 0-based index)
        matches.append((start_pos + 1, match.group(), context))

    # Print results in a readable format
    if matches:
        print(f'Found {len(matches)} occurrence(s) of {pattern_name} ({pattern}):')
        print('-' * 55)
        for position, matched_seq, context in matches:
            print(f'  Position {position:>4}: ...{context}...')
            print(f'              Matched: {matched_seq}')
            print()
    else:
        print(f'Pattern {pattern} was not found in this sequence.')

    return matches


print('find_motif() function defined and ready to use.')

In [ ]:
print('=== Searching for Sequence Motifs in Hemoglobin Alpha Chain ===')
print()

# -----------------------------------------------------------------------
# Motif 1: FPH  (Phenylalanine - Proline - Histidine)
# This tripeptide appears in the CD loop of globin proteins, just
# adjacent to the heme-binding pocket. It is conserved across
# vertebrate hemoglobins and myoglobins — a hallmark of the globin family.
# -----------------------------------------------------------------------
print('MOTIF 1: Heme-binding region — FPH (Phe-Pro-His)')
print('Found in the CD loop, near the heme pocket in globin proteins')
print()
fph_matches = find_motif(alpha_sequence, 'FPH', 'heme-binding region')

# -----------------------------------------------------------------------
# Motif 2: H.H  (Histidine - any amino acid - Histidine)
# The dot (.) in a regex matches any single character.
# So H.H finds every place where two histidines are separated by
# exactly one amino acid — a signature of closely spaced functional His.
# -----------------------------------------------------------------------
print('MOTIF 2: Paired histidine pattern — H.H (His-any-His)')
print('Identifies regions where two histidines appear close together')
print()
hxh_matches = find_motif(alpha_sequence, 'H.H', 'histidine pair')

# Print a summary of findings
print('SUMMARY')
print('=' * 40)
print(f'Alpha chain length      : {len(alpha_sequence)} amino acids')
print(f'FPH motifs found        : {len(fph_matches)}')
print(f'H.H motifs found        : {len(hxh_matches)}')
print()
print('The proximal His (F8) is at position 88 (Python index 87).')
print('The distal His (E7) is at position 59 (Python index 58).')
print('Do your results match these known functional positions?')

---
## Summary

In this notebook, you:

- **Retrieved** the human hemoglobin alpha chain sequence from NCBI using BioPython's `Entrez` module
- **Counted** the frequency of each amino acid using Python's `Counter` and organized the results in a pandas DataFrame
- **Visualized** amino acid composition with bar charts, including a side-by-side comparison of the alpha and beta chains
- **Searched** for sequence motifs using the `re` module, identifying the conserved `FPH` motif near hemoglobin's heme-binding pocket
- **Connected** computation to biology: the abundance of hydrophobic residues reflects the tightly packed globin fold, while key histidines coordinate the iron-containing heme group

### Looking Ahead

In **Notebook 2**, we shift to a different protein — **ribonuclease A (RNase A, PDB: 8RAT)** — because it contains all three types of secondary structure (α-helices, β-strands, and loops) in a single compact chain, making it an ideal teaching example. We will use the DSSP algorithm to assign secondary structure to every residue and nglview to visualize the result in 3D. Hemoglobin returns as the central example in Notebooks 4 and 5.

---
## Try It Yourself: Exploring Myoglobin

**Myoglobin** is a close relative of hemoglobin — a single-chain oxygen-storage protein found in muscle tissue. While hemoglobin transports oxygen in the blood, myoglobin stores it inside muscle cells for use during intense activity. The two proteins share a common ancestor and a nearly identical 3D fold (the globin fold), but have evolved for different physiological roles.

**Your task:** Repeat the analysis from this notebook using the human myoglobin sequence.

- **Accession number:** `NP_005359.1` (human myoglobin)

### Guiding Questions

1. How does myoglobin's length compare to the hemoglobin alpha chain? What does this tell you about the two proteins?
2. Plot the amino acid frequency bar chart for myoglobin. Does the composition look similar to hemoglobin? Which amino acids differ the most?
3. Search for the `FPH` motif in myoglobin. Do you find it? Why would you expect (or not expect) this motif to be conserved between hemoglobin and myoglobin?
4. Calculate what percentage of myoglobin's sequence is histidine. Compare it to the alpha chain value. What biological role do you think histidine plays in both proteins?

> **Hint:** All the code you need is already above. Change the accession number to `NP_005359.1` and use a new variable name like `myo_record` to avoid overwriting the hemoglobin data. Try working through it yourself before looking back at earlier cells.

In [ ]:
# Write your myoglobin analysis here!
# Follow the same steps as above:
#   Step 1: Retrieve the sequence using Entrez.efetch() with id='NP_005359.1'
#   Step 2: Count amino acids with Counter()
#   Step 3: Build a DataFrame and print it
#   Step 4: Create a bar chart with matplotlib
#   Step 5: Search for the FPH motif using find_motif()
